# TrialSense AI - Data Exploration

This notebook demonstrates how to use the ClinicalTrials.gov API client to fetch and explore trial data.

In [ ]:
import sys
sys.path.append('..')

from trialsense.data.clinical_trials_client import ClinicalTrialsClient
from trialsense.data.models import TrialPhase, TrialStatus
import pandas as pd
from IPython.display import display

## 1. Fetch a Single Trial

In [ ]:
# Example: Fetch a specific trial
async with ClinicalTrialsClient() as client:
    # Use a real NCT ID
    trial = await client.get_trial("NCT04567890")
    
    print(f"Title: {trial.title}")
    print(f"Status: {trial.status}")
    print(f"Phase: {trial.phase}")
    print(f"Sponsor: {trial.sponsor.name}")
    print(f"\nConditions: {[c.name for c in trial.conditions]}")
    print(f"\nSummary: {trial.brief_summary[:200]}...")

## 2. Search for Trials

In [ ]:
# Search for lung cancer trials
async with ClinicalTrialsClient() as client:
    trials = await client.search_trials(
        condition="lung cancer",
        phase=[TrialPhase.PHASE_3],
        status=[TrialStatus.RECRUITING],
        max_results=20
    )
    
    print(f"Found {len(trials)} trials\n")
    
    # Convert to DataFrame for analysis
    trial_data = []
    for t in trials:
        trial_data.append({
            'NCT ID': t.nct_id,
            'Title': t.title[:60] + '...',
            'Phase': t.phase,
            'Status': t.status,
            'Enrollment': t.enrollment,
            'Sponsor': t.sponsor.name,
            'Locations': len(t.locations)
        })
    
    df = pd.DataFrame(trial_data)
    display(df.head(10))

## 3. Analyze Trial Characteristics

In [ ]:
import plotly.express as px

# Enrollment distribution
fig = px.histogram(df, x='Enrollment', nbins=20, 
                   title='Distribution of Trial Enrollment Sizes')
fig.show()

# Sponsor type distribution
sponsor_counts = df['Sponsor'].value_counts().head(10)
fig = px.bar(x=sponsor_counts.index, y=sponsor_counts.values,
             title='Top 10 Sponsors',
             labels={'x': 'Sponsor', 'y': 'Number of Trials'})
fig.show()

## 4. Feature Extraction for ML

Extract features that will be used for outcome prediction.

In [ ]:
from trialsense.models.outcome_predictor import OutcomePredictor

predictor = OutcomePredictor()

# Extract features from first trial
if trials:
    features = predictor._extract_features(trials[0])
    
    print("Extracted Features:")
    for key, value in features.items():
        print(f"  {key}: {value}")

## 5. Next Steps

- **Notebook 02**: Create vector embeddings for semantic search
- **Notebook 03**: Train ML model for outcome prediction
- **Notebook 04**: Test agent workflows